# StructML (236605) — Assignment 1: Feature Extraction


## Part 0 — Setup & data loading


In [1]:
# --- Colab-only install (skip if running locally with the conda env already prepared) ---
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "relbench[full]"])

In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
from collections import Counter, defaultdict
from itertools import combinations
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.auto import tqdm

from relbench.datasets import get_dataset
from relbench.tasks import get_task

import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
)
from sklearn.impute import SimpleImputer
import xgboost as xgb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

RNG = np.random.default_rng(42)
torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "device =", DEVICE)
print("sklearn", sklearn.__version__, " | xgboost", xgb.__version__)

torch 2.11.0 device = cpu
sklearn 1.6.1  | xgboost 3.2.0


In [3]:
dataset = get_dataset("rel-stack", download=False)
db = dataset.get_db()
task = get_task("rel-stack", "user-engagement", download=False)

train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

train_df = train_table.df.copy()
val_df   = val_table.df.copy()
test_df  = test_table.df.copy()

print("Tables in db :", list(db.table_dict.keys()))
for name, t in db.table_dict.items():
    print(f"  {name:12s} rows={len(t.df):>10,} pkey={t.pkey_col!s:<6} time={t.time_col!s:<14} fkeys={t.fkey_col_to_pkey_table}")

print()
print("Target tables:")
print(f"  train: shape={train_df.shape}, unique users={train_df.OwnerUserId.nunique():,}, "
      f"timestamps {train_df.timestamp.min().date()}..{train_df.timestamp.max().date()}")
print(f"  val  : shape={val_df.shape},  unique users={val_df.OwnerUserId.nunique():,}, "
      f"timestamps {val_df.timestamp.min().date()}..{val_df.timestamp.max().date()}")
print(f"  test : shape={test_df.shape},  unique users={test_df.OwnerUserId.nunique():,}, "
      f"timestamps {test_df.timestamp.min().date()}..{test_df.timestamp.max().date()}")

print("\nClass balance:")
print(" train:", train_df["contribution"].value_counts(normalize=True).round(4).to_dict())
print(" val  :", val_df["contribution"].value_counts(normalize=True).round(4).to_dict())

Loading Database object from /home/wertheizer/.cache/relbench/rel-stack/db...


Done in 14.28 seconds.


Tables in db : ['badges', 'comments', 'postHistory', 'postLinks', 'posts', 'users', 'votes']
  badges       rows=   463,463 pkey=Id     time=Date           fkeys={'UserId': 'users'}
  comments     rows=   623,967 pkey=Id     time=CreationDate   fkeys={'UserId': 'users', 'PostId': 'posts'}
  postHistory  rows= 1,175,368 pkey=Id     time=CreationDate   fkeys={'PostId': 'posts', 'UserId': 'users'}
  postLinks    rows=    77,337 pkey=Id     time=CreationDate   fkeys={'PostId': 'posts', 'RelatedPostId': 'posts'}
  posts        rows=   333,893 pkey=Id     time=CreationDate   fkeys={'OwnerUserId': 'users', 'ParentId': 'posts'}
  users        rows=   255,360 pkey=Id     time=CreationDate   fkeys={}
  votes        rows= 1,317,876 pkey=Id     time=CreationDate   fkeys={'PostId': 'posts', 'UserId': 'users'}

Target tables:
  train: shape=(1360850, 3), unique users=83,531, timestamps 2009-04-16..2020-07-02
  val  : shape=(85838, 3),  unique users=85,838, timestamps 2020-10-01..2020-10-01
  test : 

## Part 1 — Tabular Feature Extraction
### Q1. Three hand-crafted join-aggregate features


In [4]:
def build_q1_manual_features(target_df: pd.DataFrame) -> pd.DataFrame:
    """Return target_df with the three manual features appended."""
    out = target_df.copy()

    # posts count
    posts = db.table_dict["posts"].df[["OwnerUserId"]].dropna()
    posts_per_user = (
        posts.groupby("OwnerUserId").size().rename("q1_posts_total").astype("float64")
    )
    out = out.merge(posts_per_user, left_on="OwnerUserId", right_index=True, how="left")
    out["q1_posts_total"] = out["q1_posts_total"].fillna(0.0)


    # comments count
    comments = db.table_dict["comments"].df[["UserId"]].dropna()
    comments_per_user = (
        comments.groupby("UserId").size().rename("q1_comments_total").astype("float64")
    )
    out = out.merge(comments_per_user, left_on="OwnerUserId", right_index=True, how="left")
    out["q1_comments_total"] = out["q1_comments_total"].fillna(0.0)


    # badges count
    badges = db.table_dict["badges"].df[["UserId"]].dropna()
    badges_per_user = (
        badges.groupby("UserId").size().rename("q1_badge_count").astype("float64")
    )
    out = out.merge(badges_per_user, left_on="OwnerUserId", right_index=True, how="left")
    out["q1_badge_count"] = out["q1_badge_count"].fillna(0.0)

    return out

_q1_sample = build_q1_manual_features(train_df.head(2000))
print("Q1 sample stats:")
print(_q1_sample[["q1_posts_total", "q1_comments_total", "q1_badge_count"]].describe().round(2))



Q1 sample stats:
       q1_posts_total  q1_comments_total  q1_badge_count
count         2000.00            2000.00         2000.00
mean            33.76              99.56           16.62
std            202.30             877.49           62.31
min              0.00               0.00            0.00
25%              1.00               0.00            2.00
50%              2.00               2.00            4.00
75%              7.00               9.00           10.00
max           4672.00           23771.00         1384.00


### Q2. Automatic join-aggregate feature generation


In [5]:
# ---- Build the PK-FK relationship graph -------------------------------------------------
# Nodes are tables. Each FK column gives a (parent, child, fk_col) edge.
SCHEMA_FKEYS: List[Tuple[str, str, str]] = []   # (parent_table, child_table, fk_col_in_child)
for child, t in db.table_dict.items():
    for fk_col, parent in t.fkey_col_to_pkey_table.items():
        SCHEMA_FKEYS.append((parent, child, fk_col))

print("PK-FK edges in db:")
for p, c, f in SCHEMA_FKEYS:
    print(f"  {p:12s} <--({f})-- {c}")

# The target table has its own FK: target.OwnerUserId -> users.Id .
# We model it as an extra edge so BFS can start from "target".
TARGET_FK_COL = "OwnerUserId"      # column in target that links to users.Id
TARGET_FK_PARENT = "users"
SCHEMA_FKEYS.append((TARGET_FK_PARENT, "_target_", TARGET_FK_COL))

PK-FK edges in db:
  users        <--(UserId)-- badges
  users        <--(UserId)-- comments
  posts        <--(PostId)-- comments
  posts        <--(PostId)-- postHistory
  users        <--(UserId)-- postHistory
  posts        <--(PostId)-- postLinks
  posts        <--(RelatedPostId)-- postLinks
  users        <--(OwnerUserId)-- posts
  posts        <--(ParentId)-- posts
  posts        <--(PostId)-- votes
  users        <--(UserId)-- votes


In [6]:
def column_dtype_kind(series: pd.Series) -> str:
    """Map a pandas dtype to one of {'numeric','categorical','timestamp'}."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return "timestamp"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    return "categorical"

AGGS_FOR_KIND = {
    "numeric":     ["mean", "count", "sum"],
    "categorical": ["count"],
    "timestamp":   ["min", "max"],
}

In [7]:
def enumerate_join_paths(max_depth: int) -> List[List[Tuple[str, str, str]]]:
    """Enumerate all join paths of length 1..max_depth starting from the (synthetic)
    target table. Each path is a list of edges  (from_table, to_table, join_col_pair).

    We treat the relationship graph as undirected for navigation, but remember the
    direction of each FK so we know which column joins which.
    """
    # Build adjacency: table -> list of (neighbour_table, fk_parent, fk_child, fk_col)
    adj: Dict[str, List[Tuple[str, str, str, str]]] = defaultdict(list)
    for parent, child, fk_col in SCHEMA_FKEYS:
        # forward: parent --(pk -> fk_col on child)-- child
        adj[parent].append((child, parent, child, fk_col))
        # backward: child --(fk_col -> pk on parent)-- parent
        adj[child].append((parent, parent, child, fk_col))

    paths: List[List[Tuple[str, str, str]]] = []
    # BFS storing (current_table, visited_set, path_so_far)
    frontier = [("_target_", {"_target_"}, [])]
    for depth in range(max_depth):
        next_frontier = []
        for cur, visited, path in frontier:
            for nbr, parent, child, fk_col in adj[cur]:
                if nbr in visited:
                    continue
                edge = (cur, nbr, f"{parent}.Id={child}.{fk_col}")
                new_path = path + [edge]
                paths.append(new_path)
                next_frontier.append((nbr, visited | {nbr}, new_path))
        frontier = next_frontier
    return paths

def describe_path(path) -> str:
    return " -> ".join([path[0][0]] + [e[1] for e in path])

In [8]:
VAL_TIMESTAMP = pd.Timestamp("2020-10-01")  # dataset.val_timestamp

def _filter_by_time(df: pd.DataFrame, time_col, cutoff_ts) -> pd.DataFrame:
    """Keep rows where time_col <= cutoff_ts. No-op if cutoff_ts is None or time_col missing."""
    if cutoff_ts is None or time_col is None or time_col not in df.columns:
        return df
    return df[df[time_col] <= cutoff_ts]

def auto_features(target_df: pd.DataFrame,
                  max_depth: int,
                  split: str = "train",
                  verbose: bool = True):
    """Generate join-aggregate features automatically.

    For every join path of length 1..`max_depth`, we materialise the join
    `target ⟕ T_1 ⋈ ... ⋈ T_d` (left join on the first hop, natural/inner
    afterwards) and aggregate every column of the leaf table T_d, grouped by
    the originating target row.

    Temporal constraint (Q2a, edited):
      split='train': after every join, drop joined rows where the source-table
                     time_col > the target tuple's timestamp (per-row cutoff).
      split='val'  : pre-filter every source table to time_col <= val_timestamp
                     (2020-10-01) before joining.
      split='test' : no filtering (db is already pre-cut at test_timestamp).

    Returns (feature_matrix, feature_meta).
    """
    paths = enumerate_join_paths(max_depth)
    if verbose:
        print(f"Enumerated {len(paths)} join paths up to depth {max_depth}:")
        for p in paths:
            print("  ", describe_path(p))

    # Tag every target row with a unique id so we can group back to it after joining.
    target = target_df.copy()
    target["_row_id"] = np.arange(len(target))

    # For val: pre-filter every source table at the global val_timestamp.
    val_cut = VAL_TIMESTAMP if split == "val" else None
    def src_df(name: str) -> pd.DataFrame:
        return _filter_by_time(db.table_dict[name].df, db.table_dict[name].time_col, val_cut)

    users_df = src_df("users")
    users_time_col = db.table_dict["users"].time_col
    feature_meta_rows = []
    per_path_features: List[pd.DataFrame] = []

    for path in tqdm(paths, desc="paths", disable=not verbose):
        tables_seq = [edge[1] for edge in path]   # e.g. ['users'] or ['users','posts']
        depth = len(tables_seq)
        leaf_table = tables_seq[-1]
        leaf_df = src_df(leaf_table)
        leaf_time_col = db.table_dict[leaf_table].time_col
        prefix = f"d{depth}__" + "_".join(tables_seq)

        # ---- materialise the joined table for this path ----
        if depth == 1:
            # target ⟕ users   (1:1 because users.Id is a primary key)
            joined = target[["_row_id", "OwnerUserId", "timestamp"]].merge(
                users_df, left_on="OwnerUserId", right_on="Id", how="left",
            )
            # train: drop rows where users.CreationDate > target.timestamp
            if split == "train" and users_time_col is not None and users_time_col in joined.columns:
                mask = joined[users_time_col].isna() | (joined[users_time_col] <= joined["timestamp"])
                joined = joined[mask]
        elif depth == 2:
            # target ⟕ users ⋈ leaf_table
            # FK column on leaf_table that points back to users.Id:
            user_fk = next(c for c, t in db.table_dict[leaf_table].fkey_col_to_pkey_table.items()
                           if t == "users")
            joined = (
                target[["_row_id", "OwnerUserId", "timestamp"]]
                .merge(users_df[["Id"]], left_on="OwnerUserId", right_on="Id", how="left")
                .merge(leaf_df, left_on="Id", right_on=user_fk, how="inner",
                       suffixes=("", f"_{leaf_table}"))
            )
            # train: drop rows where leaf.time_col > target.timestamp
            if split == "train" and leaf_time_col is not None:
                cand_time = f"{leaf_time_col}_{leaf_table}"
                if cand_time not in joined.columns:
                    cand_time = leaf_time_col
                if cand_time in joined.columns:
                    mask = joined[cand_time] <= joined["timestamp"]
                    joined = joined[mask]
        else:
            raise NotImplementedError("Direct implementation supports max_depth ∈ {1, 2}")

        # ---- aggregate the leaf table's columns per target row ----
        # Following ACORA: skip PK and FK columns (they are keys, not features).
        # Also drop Users.AccountId per course instructions.
        leaf_pk = db.table_dict[leaf_table].pkey_col
        leaf_fks = set(db.table_dict[leaf_table].fkey_col_to_pkey_table.keys())
        skip_cols = {leaf_pk} | leaf_fks | {"AccountId"}

        grp = joined.groupby("_row_id", sort=False)
        agg_cols: Dict[str, pd.Series] = {}
        for col in leaf_df.columns:
            if col in skip_cols:
                continue
            # Find the column in `joined`. If merge added a suffix because of a name
            # collision, the leaf-table version lives under "{col}_{leaf_table}" and
            # the suffixed name takes priority. Otherwise fall back to the plain name.
            suffixed = f"{col}_{leaf_table}"
            cand = suffixed if suffixed in joined.columns else col
            if cand not in joined.columns:
                continue
            kind = column_dtype_kind(joined[cand])
            # bool columns are detected as numeric but mean/sum on them is not
            # well-defined in this context — skip and only keep count.
            if joined[cand].dtype == bool:
                kind = "categorical"
            for aggn in AGGS_FOR_KIND[kind]:
                try:
                    res = grp[cand].count() if aggn == "count" else grp[cand].agg(aggn)
                except Exception:
                    continue
                fname = f"{prefix}__{col}__{aggn}"
                agg_cols[fname] = res
                feature_meta_rows.append({
                    "feature_name": fname, "depth": depth, "path": describe_path(path),
                    "leaf_table": leaf_table, "source_column": col, "aggregation": aggn,
                })
        if agg_cols:
            per_path_features.append(pd.DataFrame(agg_cols))

    # Concatenate all per-path feature blocks (indexed by _row_id) and merge to target
    if per_path_features:
        all_feats = pd.concat(per_path_features, axis=1)
    else:
        all_feats = pd.DataFrame(index=pd.Index([], name="_row_id"))

    feature_matrix = target.merge(all_feats, left_on="_row_id", right_index=True, how="left")
    feature_matrix = feature_matrix.drop(columns=["_row_id"])

    # NaN counts -> 0 (no related rows == zero count); leave numeric/timestamp NaN as-is.
    count_cols = [c for c in feature_matrix.columns if c.endswith("__count")]
    feature_matrix[count_cols] = feature_matrix[count_cols].fillna(0)

    return feature_matrix, pd.DataFrame(feature_meta_rows)

In [9]:
t0 = time.time()
feature_matrix, feature_meta = auto_features(train_df, max_depth=2, split="train", verbose=True)
print(f"\nGenerated in {time.time()-t0:.1f}s")
print("feature_matrix shape:", feature_matrix.shape)

Enumerated 6 join paths up to depth 2:
   _target_ -> users
   _target_ -> users -> badges
   _target_ -> users -> comments
   _target_ -> users -> postHistory
   _target_ -> users -> posts
   _target_ -> users -> votes


paths:   0%|          | 0/6 [00:00<?, ?it/s]

paths:  17%|█▋        | 1/6 [00:01<00:06,  1.26s/it]

paths:  33%|███▎      | 2/6 [00:06<00:14,  3.55s/it]

paths:  50%|█████     | 3/6 [00:20<00:24,  8.30s/it]

paths:  67%|██████▋   | 4/6 [00:55<00:37, 18.79s/it]

paths:  83%|████████▎ | 5/6 [01:04<00:15, 15.34s/it]

paths: 100%|██████████| 6/6 [01:05<00:00, 10.37s/it]

paths: 100%|██████████| 6/6 [01:05<00:00, 10.87s/it]


Generated in 68.3s
feature_matrix shape: (1360850, 49)


### Q3. Feature counts at `max_depth` ∈ {1, 2, 3}


In [10]:
def count_features_for_depth(max_depth: int) -> tuple:
    """Return (n_paths, n_features, paths_by_depth_dict)."""
    paths = enumerate_join_paths(max_depth)
    by_depth = Counter()
    n_features = 0
    for p in paths:
        d = len(p)
        by_depth[d] += 1
        # leaf table = end of path
        leaf_table = p[-1][1] if d >= 1 else None
        leaf_df = db.table_dict[leaf_table].df if leaf_table != "_target_" else None
        if leaf_df is None:
            continue
        leaf_pk = db.table_dict[leaf_table].pkey_col
        leaf_fks = set(db.table_dict[leaf_table].fkey_col_to_pkey_table.keys())
        skip_cols = {leaf_pk} | leaf_fks | {"AccountId"}
        for col in leaf_df.columns:
            if col in skip_cols:
                continue
            kind = column_dtype_kind(leaf_df[col])
            if leaf_df[col].dtype == bool:
                kind = "categorical"
            n_features += len(AGGS_FOR_KIND[kind])
    return len(paths), n_features, dict(by_depth)

q3_rows = []
for d in (1, 2, 3):
    n_paths, n_feats, by = count_features_for_depth(d)
    q3_rows.append({"max_depth": d, "n_paths": n_paths, "n_features": n_feats,
                    "paths_per_depth": by})
q3_table = pd.DataFrame(q3_rows)
print(q3_table.to_string(index=False))

 max_depth  n_paths  n_features    paths_per_depth
         1        1           9             {1: 1}
         2        6          46       {1: 1, 2: 5}
         3       14         106 {1: 1, 2: 5, 3: 8}


### Q4. Shape of `feature_matrix` and how it differs from the original target table


In [11]:
print(f"Original target (train) shape: {train_df.shape}")
print(f"feature_matrix          shape: {feature_matrix.shape}")
print(f"  added columns          : {feature_matrix.shape[1] - train_df.shape[1]}")
print(f"  rows unchanged because left-outer join on (single) PK side preserves all "
      f"target rows.")

Original target (train) shape: (1360850, 3)
feature_matrix          shape: (1360850, 49)
  added columns          : 46
  rows unchanged because left-outer join on (single) PK side preserves all target rows.


### Q5.


In [12]:
examples = [
    ("d2__users_comments__Text__count",
     "Count of non-null text entries in `comments` joined to the user — i.e., **how many comments the user has written**."),
    ("d2__users_votes__CreationDate__max",
     "Maximum (latest) `CreationDate` over all `votes` cast by the user — i.e. **the timestamp of the user’s most-recent vote**."),
    ("d2__users_posts__Score__sum",
     "Sum of all `Score` values across all `posts` owned by the user — i.e., **the total net upvotes/reputation score accumulated by the user's posts**."),
]

ex_table = pd.DataFrame(
    [{"feature": n,
      "non-null %": round(feature_matrix[n].notna().mean()*100, 1) if n in feature_matrix else None,
      "mean":       round(feature_matrix[n].astype("float64").mean(), 2) if n in feature_matrix and feature_matrix[n].dtype.kind in "fi" else None,
      "meaning": m}
     for n, m in examples])

print(ex_table.to_string(index=False))

                           feature  non-null %  mean                                                                                                                                            meaning
   d2__users_comments__Text__count       100.0  7.13                                Count of non-null text entries in `comments` joined to the user — i.e., **how many comments the user has written**.
d2__users_votes__CreationDate__max         3.0   NaN                         Maximum (latest) `CreationDate` over all `votes` cast by the user — i.e. **the timestamp of the user’s most-recent vote**.
       d2__users_posts__Score__sum         NaN   NaN Sum of all `Score` values across all `posts` owned by the user — i.e., **the total net upvotes/reputation score accumulated by the user's posts**.


### Q6. Missing values in extracted features


In [13]:
new_cols = [c for c in feature_matrix.columns if c.startswith("d1__") or c.startswith("d2__")]
miss = feature_matrix[new_cols].isna().mean().sort_values(ascending=False)
print("Top 10 missingness rates among new features:")
print(miss.head(10).round(3).to_string())
print(f"\nMean missingness across all {len(new_cols)} new features: {miss.mean()*100:.1f}%")
print(f"Fraction of new features that are 100% missing: {(miss == 1).mean()*100:.1f}%")

Top 10 missingness rates among new features:
d1__users__ProfileImageUrl__mean         1.000
d2__users_votes__VoteTypeId__mean        0.970
d2__users_votes__VoteTypeId__sum         0.970
d2__users_votes__CreationDate__min       0.970
d2__users_votes__CreationDate__max       0.970
d2__users_comments__CreationDate__max    0.401
d2__users_comments__CreationDate__min    0.401
d2__users_badges__Date__min              0.088
d2__users_badges__Date__max              0.088
d2__users_badges__Class__mean            0.088

Mean missingness across all 46 new features: 13.7%
Fraction of new features that are 100% missing: 2.2%


## Q7 — Train DecisionTree, XGBoost, Logistic Regression and a PyTorch NN


In [14]:
print("Computing features for val/test ...")
val_feat,  _ = auto_features(val_df,  max_depth=2, split="val",  verbose=False)
test_feat, _ = auto_features(test_df, max_depth=2, split="test", verbose=False)
train_feat = feature_matrix  # already computed above

print("train_feat:", train_feat.shape, "| val_feat:", val_feat.shape, "| test_feat:", test_feat.shape)

Computing features for val/test ...


train_feat: (1360850, 49) | val_feat: (85838, 49) | test_feat: (88137, 48)


In [15]:
def to_xy(df: pd.DataFrame, label_col: str = "contribution"):
    """Convert a feature-matrix-style df to (X, y).
    Drops non-feature columns; encodes timestamp aggregations to int64 nanoseconds.
    """
    drop_cols = {"OwnerUserId", "timestamp", "contribution"}
    feat_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feat_cols].copy()
    # encode timestamp-typed columns
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            # convert to seconds since epoch (float to allow NaN)
            X[c] = X[c].astype("int64").where(X[c].notna(), np.nan).astype("float64") / 1e9
        elif X[c].dtype == "object":
            # rare: object columns left over -> drop
            X = X.drop(columns=[c])
            feat_cols.remove(c)
    X = X.astype("float64")
    y = df[label_col].astype("int64").values if label_col in df.columns else None
    return X, y, feat_cols

X_tr, y_tr, feat_cols = to_xy(train_feat)
X_va, y_va, _         = to_xy(val_feat)
X_te, _,    _         = to_xy(test_feat)
print("Shapes:", X_tr.shape, X_va.shape, X_te.shape)
print("Dtypes ok:", X_tr.dtypes.value_counts().to_dict())
print("Train missingness:", X_tr.isna().mean().mean().round(3))

Shapes: (1360850, 46) (85838, 46) (88137, 46)
Dtypes ok: {dtype('float64'): 46}
Train missingness: 0.137


In [16]:
imputer = SimpleImputer(strategy="median").fit(X_tr)
X_tr_imp = imputer.transform(X_tr)
X_va_imp = imputer.transform(X_va)
X_te_imp = imputer.transform(X_te)

scaler = StandardScaler().fit(X_tr_imp)
X_tr_sc = scaler.transform(X_tr_imp)
X_va_sc = scaler.transform(X_va_imp)
X_te_sc = scaler.transform(X_te_imp)

print("Preprocessing done")

Preprocessing done


In [17]:
def evaluate(y_true, y_pred, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def eval_split(model, X, y, threshold=0.5):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= threshold).astype(int)
    return evaluate(y, pred, prob)

### Decision Tree — hyperparameter sweep


In [18]:
dt_results = []
for max_depth in [3, 5, 10, 20]:
    for min_samples_leaf in [50, 500]:
        clf = DecisionTreeClassifier(max_depth=max_depth,
                                     min_samples_leaf=min_samples_leaf,
                                     class_weight="balanced",
                                     random_state=42)
        clf.fit(X_tr_imp, y_tr)
        tr = eval_split(clf, X_tr_imp, y_tr)
        va = eval_split(clf, X_va_imp, y_va)
        dt_results.append({"max_depth": str(max_depth), "min_samples_leaf": min_samples_leaf,
                           **{f"val_{k}": round(v, 4) for k, v in va.items()},
                           **{f"tr_{k}": round(v, 4) for k, v in tr.items()}})
dt_df = pd.DataFrame(dt_results).sort_values("val_roc_auc", ascending=False)
print(dt_df.to_string(index=False))
dt_best_cfg = dt_df.iloc[0][["max_depth", "min_samples_leaf"]].to_dict()
dt_best = DecisionTreeClassifier(
    max_depth=None if dt_best_cfg["max_depth"] == "None" else int(dt_best_cfg["max_depth"]),
    min_samples_leaf=int(dt_best_cfg["min_samples_leaf"]),
    class_weight="balanced", random_state=42).fit(X_tr_imp, y_tr)
dt_metrics = {"train": eval_split(dt_best, X_tr_imp, y_tr),
              "val":   eval_split(dt_best, X_va_imp, y_va)}
print("Best DT cfg:", dt_best_cfg)
print("Best DT metrics:", dt_metrics)

max_depth  min_samples_leaf  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1
       20               500        0.7133       0.8721         0.0784      0.8557  0.1436       0.7673      0.8546        0.1486     0.7729 0.2492
       10               500        0.7121       0.8715         0.0791      0.8685  0.1449       0.7782      0.8360        0.1500     0.7364 0.2492
       10                50        0.7238       0.8567         0.0795      0.8349  0.1452       0.7860      0.8405        0.1550     0.7372 0.2561
        5               500        0.6718       0.8523         0.0708      0.8814  0.1311       0.7740      0.8048        0.1417     0.6966 0.2355
        5                50        0.6718       0.8523         0.0708      0.8814  0.1311       0.7740      0.8048        0.1417     0.6966 0.2355
       20                50        0.7574       0.8283         0.0849      0.7814  0.1532       0.8023      0.9041    

Best DT cfg: {'max_depth': '20', 'min_samples_leaf': 500}
Best DT metrics: {'train': {'accuracy': 0.7672814784877099, 'roc_auc': np.float64(0.8546187598032231), 'precision': 0.14858047284597883, 'recall': 0.7728609232578654, 'f1': 0.2492443727997914}, 'val': {'accuracy': 0.713273841422214, 'roc_auc': np.float64(0.8721277806350392), 'precision': 0.07836061837657159, 'recall': 0.8556615512235587, 'f1': 0.143572969587306}}


### XGBoost — hyperparameter sweep


In [19]:
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pos_weight = neg / pos
xgb_results = []
for max_depth in [4, 6]:
    for n_estimators in [300]:
        for lr in [0.05, 0.1]:
            clf = xgb.XGBClassifier(
                max_depth=max_depth, n_estimators=n_estimators, learning_rate=lr,
                subsample=0.9, colsample_bytree=0.9,
                scale_pos_weight=scale_pos_weight, n_jobs=-1,
                eval_metric="auc", random_state=42,
                tree_method="hist",
            )
            clf.fit(X_tr, y_tr)
            tr = eval_split(clf, X_tr, y_tr)
            va = eval_split(clf, X_va, y_va)
            xgb_results.append({"max_depth": max_depth, "n_estimators": n_estimators, "lr": lr,
                                **{f"val_{k}": round(v, 4) for k, v in va.items()},
                                **{f"tr_{k}": round(v, 4) for k, v in tr.items()}})
xgb_df = pd.DataFrame(xgb_results).sort_values("val_roc_auc", ascending=False)
print(xgb_df.to_string(index=False))
xgb_best_cfg = xgb_df.iloc[0][["max_depth", "n_estimators", "lr"]].to_dict()
xgb_best = xgb.XGBClassifier(
    max_depth=int(xgb_best_cfg["max_depth"]),
    n_estimators=int(xgb_best_cfg["n_estimators"]),
    learning_rate=float(xgb_best_cfg["lr"]),
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight, n_jobs=-1,
    eval_metric="auc", random_state=42, tree_method="hist",
).fit(X_tr, y_tr)
xgb_metrics = {"train": eval_split(xgb_best, X_tr, y_tr),
               "val":   eval_split(xgb_best, X_va, y_va)}
print("Best XGB cfg:", xgb_best_cfg)
print("Best XGB metrics:", xgb_metrics)

 max_depth  n_estimators   lr  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1
         4           300 0.05        0.7315       0.8809         0.0832      0.8548  0.1517       0.7909      0.8450        0.1575     0.7316 0.2592
         4           300 0.10        0.7330       0.8800         0.0840      0.8594  0.1531       0.7930      0.8509        0.1601     0.7398 0.2633
         6           300 0.05        0.7393       0.8795         0.0853      0.8511  0.1550       0.7999      0.8587        0.1663     0.7487 0.2722
         6           300 0.10        0.7533       0.8736         0.0879      0.8299  0.1589       0.8072      0.8699        0.1740     0.7628 0.2834


Best XGB cfg: {'max_depth': 4.0, 'n_estimators': 300.0, 'lr': 0.05}
Best XGB metrics: {'train': {'accuracy': 0.7909181761399126, 'roc_auc': np.float64(0.8449765380206375), 'precision': 0.1574669109387706, 'recall': 0.7316377536018818, 'f1': 0.2591567485373417}, 'val': {'accuracy': 0.7314592604673921, 'roc_auc': np.float64(0.8809265254373371), 'precision': 0.08323237218318391, 'recall': 0.8548320199087516, 'f1': 0.15169469694181725}}


### Logistic Regression — hyperparameter sweep


In [20]:
lr_results = []
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = LogisticRegression(C=C, class_weight="balanced",
                             max_iter=2000, solver="lbfgs", n_jobs=-1)
    clf.fit(X_tr_sc, y_tr)
    tr = eval_split(clf, X_tr_sc, y_tr)
    va = eval_split(clf, X_va_sc, y_va)
    lr_results.append({"C": C,
                       **{f"val_{k}": round(v, 4) for k, v in va.items()},
                       **{f"tr_{k}": round(v, 4) for k, v in tr.items()}})
lr_df = pd.DataFrame(lr_results).sort_values("val_roc_auc", ascending=False)
print(lr_df.to_string(index=False))
lr_best_C = float(lr_df.iloc[0]["C"])
lr_best = LogisticRegression(C=lr_best_C, class_weight="balanced",
                             max_iter=2000, solver="lbfgs", n_jobs=-1).fit(X_tr_sc, y_tr)
lr_metrics = {"train": eval_split(lr_best, X_tr_sc, y_tr),
              "val":   eval_split(lr_best, X_va_sc, y_va)}
print("Best LR C:", lr_best_C)
print("Best LR metrics:", lr_metrics)

    C  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1
 0.01        0.7870       0.8299         0.0920      0.7424  0.1637       0.8021      0.8082        0.1529     0.6517 0.2477
 1.00        0.7877       0.8293         0.0923      0.7424  0.1642       0.8021      0.8081        0.1529     0.6515 0.2476
 0.10        0.7877       0.8292         0.0923      0.7424  0.1642       0.8021      0.8081        0.1528     0.6514 0.2475
10.00        0.7880       0.8292         0.0924      0.7420  0.1643       0.8021      0.8081        0.1528     0.6513 0.2476


Best LR C: 0.01
Best LR metrics: {'train': {'accuracy': 0.8020707645956571, 'roc_auc': np.float64(0.8081613031722064), 'precision': 0.15287107408378048, 'recall': 0.6517494854454572, 'f1': 0.24765373614587058}, 'val': {'accuracy': 0.7869708054707705, 'roc_auc': np.float64(0.8298708651310021), 'precision': 0.09200719609354921, 'recall': 0.7424305267523849, 'f1': 0.16372450379584744}}


### NN in PyTorch


In [21]:
class MLP(nn.Module):
    def __init__(self, in_dim, hid=(128, 64), p=0.3):
        super().__init__()
        layers = []
        last = in_dim
        for h in hid:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(p)]
            last = h
        layers += [nn.Linear(last, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(X_tr_sc, y_tr, X_va_sc, y_va, epochs=5, batch_size=4096, lr=1e-3, p=0.3,
              hid=(128,64), seed=42, verbose=True):
    torch.manual_seed(seed)
    Xtr = torch.tensor(X_tr_sc, dtype=torch.float32)
    ytr = torch.tensor(y_tr,   dtype=torch.float32)
    Xva = torch.tensor(X_va_sc, dtype=torch.float32)
    yva = torch.tensor(y_va,   dtype=torch.float32)
    pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
                              dtype=torch.float32)
    model = MLP(Xtr.shape[1], hid=hid, p=p).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    ds = TensorDataset(Xtr, ytr)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0,
                    pin_memory=False)
    best_auc = 0.0
    best_state = None
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            va_logits = model(Xva.to(DEVICE)).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logits))
        auc = roc_auc_score(y_va, va_prob)
        if verbose:
            print(f"  epoch {ep+1}: val AUC = {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_auc

def mlp_predict(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    prob = 1 / (1 + np.exp(-logits))
    pred = (prob >= 0.5).astype(int)
    return pred, prob

# Small architecture sweep
nn_results = []
nn_models = {}
for hid in [(64,), (128, 64), (256, 128, 64)]:
    print(f"\nNN hid={hid}")
    model, _ = train_mlp(X_tr_sc, y_tr, X_va_sc, y_va, epochs=5, hid=hid)
    pred_tr, prob_tr = mlp_predict(model, X_tr_sc)
    pred_va, prob_va = mlp_predict(model, X_va_sc)
    tr = evaluate(y_tr, pred_tr, prob_tr)
    va = evaluate(y_va, pred_va, prob_va)
    nn_results.append({"hid": str(hid),
                       **{f"val_{k}": round(v,4) for k,v in va.items()},
                       **{f"tr_{k}": round(v,4) for k,v in tr.items()}})
    nn_models[str(hid)] = model
nn_df = pd.DataFrame(nn_results).sort_values("val_roc_auc", ascending=False)
print("\nNN sweep:")
print(nn_df.to_string(index=False))
nn_best = nn_models[nn_df.iloc[0]["hid"]]
nn_best_hid = nn_df.iloc[0]["hid"]
pred_tr, prob_tr = mlp_predict(nn_best, X_tr_sc)
pred_va, prob_va = mlp_predict(nn_best, X_va_sc)
nn_metrics = {"train": evaluate(y_tr, pred_tr, prob_tr),
              "val":   evaluate(y_va, pred_va, prob_va)}
print("Best NN hid:", nn_best_hid, "metrics:", nn_metrics)


NN hid=(64,)


  epoch 1: val AUC = 0.8637


  epoch 2: val AUC = 0.8635


  epoch 3: val AUC = 0.8626


  epoch 4: val AUC = 0.8630


  epoch 5: val AUC = 0.8645


... (10 outputs left out)

  epoch 4: val AUC = 0.8772


  epoch 5: val AUC = 0.8725



NN sweep:
           hid  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1
(256, 128, 64)        0.7129       0.8772         0.0790      0.8656  0.1449       0.7823      0.8404        0.1518     0.7317 0.2515
     (128, 64)        0.7319       0.8712         0.0827      0.8465  0.1506       0.7869      0.8399        0.1538     0.7249 0.2537
         (64,)        0.7352       0.8645         0.0822      0.8287  0.1495       0.7785      0.8334        0.1483     0.7237 0.2462


Best NN hid: (256, 128, 64) metrics: {'train': {'accuracy': 0.7822581474813536, 'roc_auc': np.float64(0.8403632439248317), 'precision': 0.1518188861571874, 'recall': 0.7317259629520729, 'f1': 0.25146390877536895}, 'val': {'accuracy': 0.7129359957128545, 'roc_auc': np.float64(0.877203321682936), 'precision': 0.07904105438569914, 'recall': 0.8656159270012443, 'f1': 0.14485511018566719}}


# Hyper Parameters Discussion


### Combined results table (Q7)


In [22]:
def fmt_pair(tr, va, k):
    return f"{tr[k]:.3f} / {va[k]:.3f}"

q7_table = pd.DataFrame(
    [
        {"model": "Decision Tree",       **{m: fmt_pair(dt_metrics["train"],  dt_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
        {"model": "XGBoost",             **{m: fmt_pair(xgb_metrics["train"], xgb_metrics["val"], m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
        {"model": "Logistic Regression", **{m: fmt_pair(lr_metrics["train"],  lr_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
        {"model": "MLP (PyTorch)",       **{m: fmt_pair(nn_metrics["train"],  nn_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
    ]
)
print("Q7 — Train / Val metrics (best variant per model):")
print(q7_table.to_string(index=False))

Q7 — Train / Val metrics (best variant per model):
              model      accuracy       roc_auc     precision        recall            f1
      Decision Tree 0.767 / 0.713 0.855 / 0.872 0.149 / 0.078 0.773 / 0.856 0.249 / 0.144
            XGBoost 0.791 / 0.731 0.845 / 0.881 0.157 / 0.083 0.732 / 0.855 0.259 / 0.152
Logistic Regression 0.802 / 0.787 0.808 / 0.830 0.153 / 0.092 0.652 / 0.742 0.248 / 0.164
      MLP (PyTorch) 0.782 / 0.713 0.840 / 0.877 0.152 / 0.079 0.732 / 0.866 0.251 / 0.145


## Q8 — Evaluation measures


## Q9 — Discussion of results


## Q10 — How feature importance is computed in each model


## Q11 — Top-10 features per model


In [23]:
def top_features(model, feature_names, k=10, attr="auto"):
    """Return top-k (feature_name, importance) pairs."""
    if attr == "auto":
        if hasattr(model, "feature_importances_"):
            imp = np.asarray(model.feature_importances_)
        elif hasattr(model, "coef_"):
            imp = np.abs(np.asarray(model.coef_)).ravel()
        else:
            raise ValueError("no importance attribute")
    else:
        imp = np.abs(np.asarray(getattr(model, attr))).ravel()
    order = np.argsort(imp)[::-1][:k]
    return [(feature_names[i], float(imp[i])) for i in order]

dt_top = top_features(dt_best, feat_cols)
xgb_top = top_features(xgb_best, feat_cols)
lr_top = top_features(lr_best, feat_cols)

top_df = pd.DataFrame({
    "rank":            list(range(1, 11)),
    "DT_feature":      [f for f, _ in dt_top],
    "DT_imp":          [round(v, 4) for _, v in dt_top],
    "XGB_feature":     [f for f, _ in xgb_top],
    "XGB_imp":         [round(v, 4) for _, v in xgb_top],
    "LR_feature":      [f for f, _ in lr_top],
    "LR_|coef|":       [round(v, 4) for _, v in lr_top],
})
print("Top-10 features per model (depth-2 tabular features only):")
print(top_df.to_string(index=False))

Top-10 features per model (depth-2 tabular features only):
 rank                                 DT_feature  DT_imp                               XGB_feature  XGB_imp                                      LR_feature  LR_|coef|
    1          d2__users_posts__PostTypeId__mean  0.5903        d2__users_posts__PostTypeId__count   0.3107                  d2__users_badges__Class__count     1.8290
    2                d2__users_badges__Date__min  0.0617    d2__users_posts__ContentLicense__count   0.2613                     d2__users_badges__Date__min     1.0364
    3               d2__users_posts__Tags__count  0.0562          d2__users_posts__PostTypeId__sum   0.0963               d2__users_badges__TagBased__count     0.8934
    4   d2__users_postHistory__CreationDate__min  0.0430 d2__users_comments__ContentLicense__count   0.0423        d2__users_postHistory__CreationDate__min     0.6881
    5         d2__users_posts__CreationDate__min  0.0368                 d1__users__AboutMe__count   0.036

## Part 2 — Graph Feature Extraction


### Q12. Build the graph and compute (a) degree, (b) WL coloring, (c) two graphlet counts


In [24]:
def build_graph(cutoff_ts, inclusive: bool):
    """Build a (G, adj, filtered_tables) for tuples with time_col <= cutoff_ts
    (or < cutoff_ts when inclusive=False). cutoff_ts=None keeps all tuples."""
    G = nx.Graph()
    adj: Dict[Tuple[str,int], set] = defaultdict(set)
    filtered: Dict[str, pd.DataFrame] = {}

    for tbl_name, tbl in db.table_dict.items():
        df = tbl.df
        if cutoff_ts is not None and tbl.time_col is not None and tbl.time_col in df.columns:
            df = df[df[tbl.time_col] <= cutoff_ts] if inclusive else df[df[tbl.time_col] < cutoff_ts]
        filtered[tbl_name] = df
        pk_col = tbl.pkey_col
        ids = df[pk_col].dropna().astype("Int64").astype("int64").to_numpy()
        G.add_nodes_from(((tbl_name, int(i)) for i in ids), table=tbl_name)

    for parent, child, fk_col in SCHEMA_FKEYS:
        if child == "_target_":
            continue
        child_df = filtered[child]
        child_pk_col = db.table_dict[child].pkey_col
        parent_pk_col = db.table_dict[parent].pkey_col
        parent_ids = set(filtered[parent][parent_pk_col].dropna().astype("int64").to_numpy())
        pairs = child_df[[child_pk_col, fk_col]].dropna()
        pairs = pairs.astype({child_pk_col: "int64", fk_col: "int64"})
        pairs = pairs[pairs[fk_col].isin(parent_ids)]
        edge_list = list(zip(
            [(child, int(x))  for x in pairs[child_pk_col].to_numpy()],
            [(parent, int(x)) for x in pairs[fk_col].to_numpy()],
        ))
        G.add_edges_from(edge_list)
        for u, v in edge_list:
            adj[u].add(v)
            adj[v].add(u)
    return G, adj, filtered

print("Building 3 graphs (train < val_timestamp, val ≤ val_timestamp, test = all) ...")
t0 = time.time()
G_train, adj_train, filt_train = build_graph(cutoff_ts=VAL_TIMESTAMP, inclusive=False)
print(f"  train: nodes={G_train.number_of_nodes():,}  edges={G_train.number_of_edges():,}")
G_val,   adj_val,   filt_val   = build_graph(cutoff_ts=VAL_TIMESTAMP, inclusive=True)
print(f"  val  : nodes={G_val.number_of_nodes():,}  edges={G_val.number_of_edges():,}")
G_test,  adj_test,  filt_test  = build_graph(cutoff_ts=None,          inclusive=True)
print(f"  test : nodes={G_test.number_of_nodes():,}  edges={G_test.number_of_edges():,}")
print(f"  built all 3 in {time.time()-t0:.1f}s")

Building 3 graphs (train < val_timestamp, val ≤ val_timestamp, test = all) ...


  train: nodes=4,121,359  edges=5,641,899


  val  : nodes=4,121,747  edges=5,642,180


  test : nodes=4,247,264  edges=5,812,887
  built all 3 in 186.5s


In [25]:
users_ids = db.table_dict["users"].df["Id"].dropna().astype("int64").unique()

def degree_per_user(adj_map) -> pd.Series:
    return pd.Series({uid: len(adj_map[("users", int(uid))]) for uid in users_ids},
                     name="graph_degree", dtype="int64")

deg_train = degree_per_user(adj_train)
deg_val   = degree_per_user(adj_val)
deg_test  = degree_per_user(adj_test)
print("Degree stats over all users (test graph):")
print(deg_test.describe().round(2))
print(f"\nmean degree per split:  train={deg_train.mean():.2f}  "
      f"val={deg_val.mean():.2f}  test={deg_test.mean():.2f}")

Degree stats over all users (test graph):
count    255360.00
mean          9.83
std         258.06
min           0.00
25%           0.00
50%           1.00
75%           6.00
max       70436.00
Name: graph_degree, dtype: float64

mean degree per split:  train=9.54  val=9.54  test=9.83


In [26]:
def wl_compute(adj_map: dict, init_label_fn, K_max: int = 5):
    """Run K_max WL iterations, return {K: {node: color_id}} for K = 1..K_max."""
    colors = {n: hash(init_label_fn(n)) for n in adj_map}
    out_at = {}
    for it in tqdm(range(1, K_max + 1), desc="WL iter"):
        new_colors = {}
        for n, nbrs in adj_map.items():
            nbr_cols = tuple(sorted(colors[u] for u in nbrs))
            new_colors[n] = hash((colors[n], nbr_cols))
        colors = new_colors
        out_at[it] = dict(colors)
    return out_at

def wl_features_for(adj_map) -> pd.DataFrame:
    wl_at = wl_compute(adj_map, init_label_fn=lambda n: n[0], K_max=5)
    def user_colors(color_dict):
        return pd.Series({uid: color_dict.get(("users", int(uid)), 0) for uid in users_ids})
    df = pd.DataFrame({
        "wl_K2": user_colors(wl_at[2]).factorize()[0],
        "wl_K3": user_colors(wl_at[3]).factorize()[0],
        "wl_K5": user_colors(wl_at[5]).factorize()[0],
    }, index=users_ids)
    df.index.name = "OwnerUserId"
    return df

print("Running WL on train graph ...");  t0 = time.time()
wl_train = wl_features_for(adj_train); print(f"  done in {time.time()-t0:.1f}s")
print("Running WL on val graph ...");    t0 = time.time()
wl_val   = wl_features_for(adj_val);   print(f"  done in {time.time()-t0:.1f}s")
print("Running WL on test graph ...");   t0 = time.time()
wl_test  = wl_features_for(adj_test);  print(f"  done in {time.time()-t0:.1f}s")
print("\nDistinct WL colours among users (test graph):")
print(wl_test.nunique().to_dict())

Running WL on train graph ...


WL iter:   0%|          | 0/5 [00:00<?, ?it/s]

WL iter:  20%|██        | 1/5 [00:14<00:59, 14.90s/it]

WL iter:  40%|████      | 2/5 [00:28<00:42, 14.22s/it]

WL iter:  60%|██████    | 3/5 [00:43<00:28, 14.49s/it]

WL iter:  80%|████████  | 4/5 [00:58<00:14, 14.57s/it]

... (15 outputs left out)

WL iter:  80%|████████  | 4/5 [00:58<00:14, 14.53s/it]

WL iter: 100%|██████████| 5/5 [01:13<00:00, 14.81s/it]

WL iter: 100%|██████████| 5/5 [01:13<00:00, 14.72s/it]

  done in 78.0s

Distinct WL colours among users (test graph):
{'wl_K2': 53584, 'wl_K3': 72886, 'wl_K5': 84480}


In [27]:
def graphlet_features_for(filtered) -> pd.DataFrame:
    posts_owner = filtered["posts"][["Id", "OwnerUserId"]].dropna()
    comments_   = filtered["comments"][["UserId", "PostId"]].dropna()
    gj = comments_.merge(posts_owner.rename(columns={"Id": "PostId"}),
                         on="PostId", how="inner")
    self_comments = gj.loc[gj["UserId"] == gj["OwnerUserId"]] \
                      .groupby("UserId").size().rename("graphlet_self_comment")

    pl = filtered["postLinks"][["PostId", "RelatedPostId"]].dropna()
    gj2 = pl.merge(posts_owner.rename(columns={"Id": "PostId",
                                               "OwnerUserId": "ownerA"}), on="PostId") \
            .merge(posts_owner.rename(columns={"Id": "RelatedPostId",
                                               "OwnerUserId": "ownerB"}), on="RelatedPostId")
    linked_self_posts = gj2[gj2["ownerA"] == gj2["ownerB"]] \
                          .groupby("ownerA").size().rename("graphlet_linked_self_posts")

    out = pd.DataFrame({"graphlet_self_comment":      self_comments,
                        "graphlet_linked_self_posts": linked_self_posts}) \
            .reindex(users_ids).fillna(0).astype("int64")
    out.index.name = "OwnerUserId"
    return out

graphlet_train = graphlet_features_for(filt_train)
graphlet_val   = graphlet_features_for(filt_val)
graphlet_test  = graphlet_features_for(filt_test)
print("Graphlet feature stats (test graph, over all users):")
print(graphlet_test.describe().round(2))

Graphlet feature stats (test graph, over all users):
       graphlet_self_comment  graphlet_linked_self_posts
count              255360.00                   255360.00
mean                    0.79                        0.01
std                    14.94                        0.28
min                     0.00                        0.00
25%                     0.00                        0.00
50%                     0.00                        0.00
75%                     0.00                        0.00
max                  4174.00                       59.00


In [28]:
def graph_features_df(deg, graphlets, wl) -> pd.DataFrame:
    df = pd.concat([deg.rename("graph_degree"), graphlets, wl], axis=1)
    df.index.name = "OwnerUserId"
    return df

graph_features_train = graph_features_df(deg_train, graphlet_train, wl_train)
graph_features_val   = graph_features_df(deg_val,   graphlet_val,   wl_val)
graph_features_test  = graph_features_df(deg_test,  graphlet_test,  wl_test)
graph_cols = list(graph_features_train.columns)

def add_graph_feats(fm: pd.DataFrame, gf: pd.DataFrame) -> pd.DataFrame:
    out = fm.merge(gf, left_on="OwnerUserId", right_index=True, how="left")
    out[graph_cols] = out[graph_cols].fillna(0)
    return out

train_feat_g = add_graph_feats(train_feat, graph_features_train)
val_feat_g   = add_graph_feats(val_feat,   graph_features_val)
test_feat_g  = add_graph_feats(test_feat,  graph_features_test)
print("Augmented shapes:", train_feat_g.shape, val_feat_g.shape, test_feat_g.shape)

Augmented shapes: (1360850, 55) (85838, 55) (88137, 54)


### Q13 — Retrain all four models on the tabular + graph feature matrix


In [29]:
X_tr2, y_tr2, feat_cols2 = to_xy(train_feat_g)
X_va2, y_va2, _          = to_xy(val_feat_g)

imp2 = SimpleImputer(strategy="median").fit(X_tr2)
X_tr2_imp = imp2.transform(X_tr2)
X_va2_imp = imp2.transform(X_va2)
sc2 = StandardScaler().fit(X_tr2_imp)
X_tr2_sc = sc2.transform(X_tr2_imp)
X_va2_sc = sc2.transform(X_va2_imp)
print("Augmented preprocessing done. n_features =", X_tr2.shape[1])

# DT with best-from-before cfg
dt2 = DecisionTreeClassifier(
    max_depth=None if dt_best_cfg["max_depth"]=="None" else int(dt_best_cfg["max_depth"]),
    min_samples_leaf=int(dt_best_cfg["min_samples_leaf"]),
    class_weight="balanced", random_state=42).fit(X_tr2_imp, y_tr2)
dt2_metrics = {"train": eval_split(dt2, X_tr2_imp, y_tr2),
               "val":   eval_split(dt2, X_va2_imp, y_va2)}

# XGB
xgb2 = xgb.XGBClassifier(
    max_depth=int(xgb_best_cfg["max_depth"]),
    n_estimators=int(xgb_best_cfg["n_estimators"]),
    learning_rate=float(xgb_best_cfg["lr"]),
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight, n_jobs=-1,
    eval_metric="auc", random_state=42, tree_method="hist",
).fit(X_tr2, y_tr2)
xgb2_metrics = {"train": eval_split(xgb2, X_tr2, y_tr2),
                "val":   eval_split(xgb2, X_va2, y_va2)}

# LR
lr2 = LogisticRegression(C=lr_best_C, class_weight="balanced",
                         max_iter=2000, solver="lbfgs", n_jobs=-1).fit(X_tr2_sc, y_tr2)
lr2_metrics = {"train": eval_split(lr2, X_tr2_sc, y_tr2),
               "val":   eval_split(lr2, X_va2_sc, y_va2)}

# NN (best hid)
nn2, _ = train_mlp(X_tr2_sc, y_tr2, X_va2_sc, y_va2, epochs=5,
                   hid=eval(nn_best_hid), verbose=False)
pred_tr2, prob_tr2 = mlp_predict(nn2, X_tr2_sc)
pred_va2, prob_va2 = mlp_predict(nn2, X_va2_sc)
nn2_metrics = {"train": evaluate(y_tr2, pred_tr2, prob_tr2),
               "val":   evaluate(y_va2, pred_va2, prob_va2)}

q13_table = pd.DataFrame([
    {"model": "Decision Tree",       **{m: fmt_pair(dt2_metrics["train"],  dt2_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
    {"model": "XGBoost",             **{m: fmt_pair(xgb2_metrics["train"], xgb2_metrics["val"], m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
    {"model": "Logistic Regression", **{m: fmt_pair(lr2_metrics["train"],  lr2_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
    {"model": "MLP (PyTorch)",       **{m: fmt_pair(nn2_metrics["train"],  nn2_metrics["val"],  m) for m in ["accuracy","roc_auc","precision","recall","f1"]}},
])
print("Q13 — Train / Val metrics WITH graph features:")
print(q13_table.to_string(index=False))

Augmented preprocessing done. n_features = 52


Q13 — Train / Val metrics WITH graph features:
              model      accuracy       roc_auc     precision        recall            f1
      Decision Tree 0.864 / 0.916 0.957 / 0.827 0.260 / 0.167 0.934 / 0.505 0.407 / 0.251
            XGBoost 0.883 / 0.916 0.959 / 0.873 0.290 / 0.188 0.930 / 0.599 0.442 / 0.286
Logistic Regression 0.833 / 0.794 0.899 / 0.808 0.203 / 0.095 0.798 / 0.740 0.323 / 0.168
      MLP (PyTorch) 0.868 / 0.859 0.954 / 0.865 0.265 / 0.129 0.926 / 0.703 0.412 / 0.219


### Q14 — Top-10 features per model (now with graph features available)


In [30]:
dt2_top = top_features(dt2, feat_cols2)
xgb2_top = top_features(xgb2, feat_cols2)
lr2_top = top_features(lr2, feat_cols2)

graph_feat_set = set(graph_cols)
def annotate(rank_top, k=10):
    rows = []
    for f, v in rank_top[:k]:
        kind = "graph" if f in graph_feat_set else "tabular"
        rows.append((f, round(v,4), kind))
    return rows

top2 = pd.DataFrame({
    "rank":           list(range(1, 11)),
    "DT_feature":     [r[0] for r in annotate(dt2_top)],
    "DT_kind":        [r[2] for r in annotate(dt2_top)],
    "XGB_feature":    [r[0] for r in annotate(xgb2_top)],
    "XGB_kind":       [r[2] for r in annotate(xgb2_top)],
    "LR_feature":     [r[0] for r in annotate(lr2_top)],
    "LR_kind":        [r[2] for r in annotate(lr2_top)],
})
print("Top-10 features (graph in **bold** annotation):")
print(top2.to_string(index=False))

# How many graph features made each model's top-10?
print()
for name, lst in [("DT", dt2_top), ("XGB", xgb2_top), ("LR", lr2_top)]:
    g = sum(1 for f, _ in lst[:10] if f in graph_feat_set)
    print(f"  {name:>3}: graph features in top-10 = {g}")

Top-10 features (graph in **bold** annotation):
 rank                                     DT_feature DT_kind                                     XGB_feature XGB_kind                                    LR_feature LR_kind
    1             d2__users_votes__CreationDate__max tabular                                    graph_degree    graph            d2__users_votes__CreationDate__max tabular
    2 d2__users_postHistory__PostHistoryTypeId__mean tabular      d2__users_postHistory__RevisionGUID__count  tabular                   d2__users_badges__Date__max tabular
    3   d2__users_postHistory__ContentLicense__count tabular                           graphlet_self_comment    graph    d2__users_comments__UserDisplayName__count tabular
    4     d2__users_comments__UserDisplayName__count tabular d2__users_postHistory__PostHistoryTypeId__count  tabular                                  graph_degree   graph
    5                  d2__users_badges__Name__count tabular                  d2__users_badg

## Q15 — Summary (conclusion paragraph for the report)


## Q16 — AI-tool usage
